# TP1 - Du texte brut au modele bigramme

Ce notebook transforme Tiny Shakespeare en exemples supervises, entraine une table de logits et compare une generation avant/apres apprentissage.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
DATA_PATH = Path('input.txt')
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not DATA_PATH.exists():
    urlretrieve(URL, DATA_PATH)
text = DATA_PATH.read_text(encoding='utf-8')
print(f'Corpus: {len(text):,} caracteres')
print(repr(text[:300]))

Un token est ici un caractere. L'encodage UTF-8 permet de decoder correctement les octets en texte Unicode. Le corpus est une longue sequence, pas une liste d'exemples deja separes.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)
message = 'hii there'
print('vocab_size =', vocab_size)
print(encode(message))
print(decode(encode(message)))
assert decode(encode(message)) == message

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
block_size, batch_size = 8, 32
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    print(x[:t + 1].tolist(), '->', y[t].item())

def get_batch(split):
    source = train_data if split == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    xb = torch.stack([source[i:i + block_size] for i in starts])
    yb = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return xb, yb

xb, yb = get_batch('train')
print('batch shapes:', xb.shape, yb.shape)

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.reshape(B * T, C), targets.reshape(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, 200)[0].tolist()))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses = []
for step in range(1000):
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step in (0, 99, 999):
        print(f'step {step + 1}: loss={loss.item():.4f}')

print(decode(model.generate(context, 500)[0].tolist()))

### Questions et reponses

1. **Quelle est l'unite d'un token ?** Ici, c'est un caractere (espace, retour a la ligne et ponctuation compris). Le corpus n'est pas separe en exemples: les fenetres sont fabriquees par `get_batch`.
2. **Pourquoi `encoding='utf-8'` ?** Pour decoder correctement les octets du fichier en texte Unicode et eviter les caracteres corrompus.
3. **Pourquoi trier `chars` ?** Le tri rend l'association caractere-identifiant deterministe entre deux executions. `stoi` et `itos` sont inverses, donc `decode(encode(message)) == message` valide la bijection.
4. **Pourquoi decaler `x` et `y` ?** `x[t]` est le contexte observe et `y[t]` est le caractere suivant a predire. Avec `block_size=8`, un bloc produit 8 predictions.
5. **Pourquoi separer train et validation ?** La validation mesure la generalisation sur la fin du corpus, sans melanger des informations de cette portion dans l'apprentissage.
6. **Pour `t=3`, quel est le contexte ?** `x[:4]` (les quatre premiers caracteres du bloc); la cible est `y[3]`, c'est-a-dire le cinquieme caractere de la sequence originale.
7. **Pourquoi retirer `block_size` de la borne de tirage ?** Pour garantir que `i + block_size` et `i + block_size + 1` restent dans `source`. Augmenter `B` ajoute des sequences paralleles; augmenter `T` fournit un contexte plus long mais augmente le cout.
8. **Que sont `B`, `T` et `C` ?** `B` est le nombre de sequences du mini-batch, `T` leur longueur et `C` le nombre de caracteres du vocabulaire. Les logits ont donc la forme `(B,T,C)`.
9. **Pourquoi une table `C x C` dans le bigramme ?** Chaque caractere d'entree selectionne une ligne contenant un score pour chacun des `C` caracteres suivants. Dans le TP4, cette idee est remplacee par un embedding `C x n_embd` puis `lm_head` `n_embd x C`.
10. **Comment les logits deviennent-ils des probabilites ?** `softmax` normalise les scores sur la derniere dimension; `cross_entropy` applique cette operation de maniere stable et compare la distribution a la cible.
11. **Pourquoi garder seulement le dernier pas pendant la generation ?** Les pas precedents ont deja ete generes; seul `logits[:, -1, :]` donne la distribution du prochain caractere. Aplatir `B*T` permet de calculer en parallele la loss de toutes les positions.
12. **Que change l'apprentissage ?** Les lignes de la table apprennent les transitions de caracteres frequentes. La loss diminue et la sortie devient statistiquement plus proche du style du corpus, meme si 1 000 etapes ne suffisent pas a produire du texte coherent.
